In [45]:
#### BASELINE PLAYER KEY ####
# based on historical football-reference data
# includes draft data from 2025

# import packages
import os
import sys
import pandas as pd
import re
import json

# import local scripts
sys.path.append(os.path.dirname(os.path.abspath('')))
from scripts.load_data import load_data
from scripts.clean_cols import football_ref_cols, football_ref_draft_cols

# Get list of files starting with 's2' in the directory
data_dir = "../data/fpts historical"
files = [f for f in os.listdir(data_dir) if f.startswith('s2')]

# Initialize empty list to store dataframes
dfs = []

# Process each file
for file in files:
    # Extract year from filename (assuming format s2XXX where XXX is year)
    year = file[1:5]
    
    # Load the data
    df = load_data(os.path.join(data_dir, file))

    # drop the first 2 rows
    df = df.iloc[1:]
    
    # Update column names to match football_ref_cols
    df.columns = football_ref_cols
    
    # Add SEASON column
    df['SEASON'] = year
    
    # Clean player names - remove special characters
    df['PLAYER NAME'] = df['PLAYER NAME'].apply(lambda x: re.sub(r'[^a-zA-Z\s]', '', str(x)))
    
    dfs.append(df)

# Concatenate all dataframes
combined_df = pd.concat(dfs, ignore_index=True)

# Read in draft_2025.csv
draft_2025_df = pd.read_csv(data_dir + '/draft_2025.csv')

# Update column names to match football_ref_draft_cols
draft_2025_df.columns = football_ref_draft_cols

# Subset to only 'PLAYER NAME' and 'PLAYER ID', drop duplicates
draft_2025_alias_df = draft_2025_df[['PLAYER NAME', 'PLAYER ID']].drop_duplicates()

# Subset combined_df to only include 'PLAYER NAME' and 'PLAYER ID' columns
alias_df = combined_df[['PLAYER NAME', 'PLAYER ID']].drop_duplicates()
alias_df = pd.concat([alias_df, draft_2025_alias_df])

# Create player name dictionary
player_dict = alias_df.drop_duplicates('PLAYER ID').set_index('PLAYER ID')['PLAYER NAME'].to_dict()



In [50]:
player_dict

{'McCaCh01': 'Christian McCaffrey',
 'JackLa00': 'Lamar Jackson',
 'HenrDe00': 'Derrick Henry',
 'JoneAa00': 'Aaron Jones',
 'ElliEz00': 'Ezekiel Elliott',
 'CookDa01': 'Dalvin Cook',
 'ThomMi05': 'Michael Thomas',
 'KelcTr00': 'Travis Kelce',
 'ChubNi00': 'Nick Chubb',
 'IngrMa01': 'Mark Ingram',
 'EkelAu00': 'Austin Ekeler',
 'PresDa01': 'Dak Prescott',
 'AndrMa00': 'Mark Andrews',
 'WilsRu00': 'Russell Wilson',
 'GodwCh00': 'Chris Godwin',
 'KittGe00': 'George Kittle',
 'WatsDe00': 'Deshaun Watson',
 'CarsCh00': 'Chris Carson',
 'GollKe00': 'Kenny Golladay',
 'WallDa01': 'Darren Waller',
 'BarkSa00': 'Saquon Barkley',
 'ErtzZa00': 'Zach Ertz',
 'MixoJo00': 'Joe Mixon',
 'GurlTo01': 'Todd Gurley',
 'CookJa02': 'Jared Cook',
 'KuppCo00': 'Cooper Kupp',
 'JoneJu02': 'Julio Jones',
 'ParkDe01': 'DeVante Parker',
 'FourLe00': 'Leonard Fournette',
 'WinsJa00': 'Jameis Winston',
 'HoopAu00': 'Austin Hooper',
 'CoopAm00': 'Amari Cooper',
 'EvanMi00': 'Mike Evans',
 'BrowAJ00': 'AJ Brown',
 

In [46]:
#### FULL PLAYER KEY ####
# includes fantasy rankings and aliases from multiple sources 

# Load each file in data_path into a distinct DataFrame using load_data.

data_path = "../data/rankings current/update/"

# List all files in the data_path directory
files = [f for f in os.listdir(data_path) if not f.startswith('.')]

# Sort files for reproducibility (optional, but helps with mapping)
files.sort()

# Load each file into a DataFrame using load_data
# Step 1: Create a lookup table mapping comment names to generalized file names.
# We'll use the first 12 characters of each file name (adjust as needed for your files).
# The keys are the comment names, the values are the generalized file names.

lookup_keys = [
    "fpts",
    "fantasypros",
    "jj",
    "draftshark_adp",
    "draftshark_rank",
    "hayden_winks"
]

# Generalize file names by taking the first 12 characters (adjust if needed)
generalized_files = [f[:11] for f in files[:len(lookup_keys)]]

# Build the lookup dictionary
file_lookup = dict(zip(lookup_keys, generalized_files))

# Step 2: Use the lookup to read in each file and assign to a variable named after the key.
# We'll use globals() to dynamically assign variable names.

# Create a dictionary of DataFrames, keyed by lookup_key, with values from load_data
dataframes = {}
for key, gen_file in file_lookup.items():
    # Find the actual file in files that starts with gen_file
    matched_file = next((f for f in files if f.startswith(gen_file)), None)
    if matched_file:
        dataframes[key] = load_data(os.path.join(data_path, matched_file))
    else:
        raise ValueError(f"No file found for key '{key}' with prefix '{gen_file}'.")

# Now you have a dictionary: dataframes['fpts'], dataframes['fantasypros'], etc.

In [47]:
# update col names in dataframes dict

from scripts.clean_cols import cols_dict

for key, df in dataframes.items():
    if key in cols_dict:
        df.columns = cols_dict[key]


In [48]:
import re

# 1. Remove special characters from PLAYER NAME in each DataFrame
for df in dataframes.values():
    if 'PLAYER NAME' in df.columns:
        # Remove +, *, ., ', and any other non-alphanumeric (except space)
        df['PLAYER NAME'] = df['PLAYER NAME'].astype(str).apply(lambda x: re.sub(r"[^\w\s]", "", x))

# 2. Create a list of all PLAYER NAME values from all DataFrames
all_player_names = []
for df in dataframes.values():
    if 'PLAYER NAME' in df.columns:
        all_player_names.extend(df['PLAYER NAME'].dropna().tolist())

# 3. Drop duplicates to ensure unique names
unique_player_names = list(set(all_player_names))

# Explain: 
# - We clean PLAYER NAME columns in-place for all DataFrames.
# - We gather all names into a single list, then use set() to deduplicate.
# - unique_player_names now contains all unique, cleaned player names across all DataFrames.


In [52]:
import json
from difflib import get_close_matches

# Load the player_key_dict.json file
with open("../player_key_dict_updated.json", "r") as f:
    player_key_dict = json.load(f)

def clean_name(name):
    # Remove +, *, ., ', and any other non-alphanumeric (except space)
    return re.sub(r"[^\w\s]", "", str(name))

# Build a mapping from cleaned player name to list of keys in player_key_dict
name_to_keys = {}
all_cleaned_names = []
for key, names in player_key_dict.items():
    for name in names:
        cleaned = clean_name(name)
        if cleaned not in name_to_keys:
            name_to_keys[cleaned] = []
        name_to_keys[cleaned].append(key)
        all_cleaned_names.append(cleaned)

# Track which names were matched and which were not
matched = []
not_matched = []

# Set a similarity cutoff for fuzzy matching (e.g., 0.85)
similarity_cutoff = 0.85

# For each unique player name, try to match to a key in player_key_dict using fuzzy matching
for pname in unique_player_names:
    cleaned_pname = clean_name(pname)
    # Try exact match first
    if cleaned_pname in name_to_keys:
        for key in name_to_keys[cleaned_pname]:
            if pname not in player_key_dict[key]:
                player_key_dict[key].append(pname)
        matched.append(pname)
    else:
        # Use difflib.get_close_matches for fuzzy matching
        close_matches = get_close_matches(cleaned_pname, all_cleaned_names, n=1, cutoff=similarity_cutoff)
        if close_matches:
            best_match = close_matches[0]
            for key in name_to_keys[best_match]:
                if pname not in player_key_dict[key]:
                    player_key_dict[key].append(pname)
            matched.append(pname)
            print(f"Fuzzy matched '{pname}' to '{best_match}' (key(s): {name_to_keys[best_match]})")
        else:
            print(f"No match found in player_key_dict for: '{pname}'")
            not_matched.append(pname)

# After processing all names, print summary of matches
print(f"Matched {len(matched)} out of {len(unique_player_names)} player names.")

# Optionally, save the updated player_key_dict if you want to persist changes
with open("player_key_dict_updated.json", "w") as f:
    json.dump(player_key_dict, f, indent=4)

# Explain:
# - We load and clean the player_key_dict for robust matching.
# - For each unique player name, we first try an exact match.
# - If no exact match, we use fuzzy matching (difflib.get_close_matches) with a probability cutoff.
# - If matched, we append the original name to the corresponding key's value list (if not already present).
# - If not matched, we print a message for clarity.


No match found in player_key_dict for: 'Joshua Karty'
No match found in player_key_dict for: 'Matthew Wright'
No match found in player_key_dict for: 'Washington Commanders'
No match found in player_key_dict for: 'Buffalo Bills'
No match found in player_key_dict for: 'Los Angeles Rams'
No match found in player_key_dict for: 'Jason Sanders'
No match found in player_key_dict for: 'Denver Broncos'
No match found in player_key_dict for: 'Tyler Bass'
No match found in player_key_dict for: 'Wil Lutz'
No match found in player_key_dict for: 'Isaiah Bond'
No match found in player_key_dict for: 'Scott Matlock'
No match found in player_key_dict for: 'JaQuinden Jackson'
No match found in player_key_dict for: 'JJ Galbreath'
No match found in player_key_dict for: 'Philadelphia Eagles'
No match found in player_key_dict for: 'Harrison Butker'
No match found in player_key_dict for: 'Las Vegas Raiders'
No match found in player_key_dict for: 'Nathaniel Dell'
No match found in player_key_dict for: 'Dee Esk

In [39]:
player_key_dict

{'AbanIs00': ['Israel Abanikanda'],
 'AbbrJa00': ['Jared Abbrederis'],
 'AbduAm00': ['Ameer Abdullah'],
 'AchaDe00': ['DeVon Achane'],
 'AdamDa01': ['Davante Adams'],
 'AdamJe01': ['Jerell Adams'],
 'AdamJo03': ['Josh Adams'],
 'AdamRo01': ['Rodney Adams'],
 'AddiBr01': ['Bralon Addison'],
 'AddiJo00': ['Jordan Addison'],
 'AdebQu00': ['Quincy Adeboyejo'],
 'AdkiNa00': ['Nate Adkins'],
 'AghoNe00': ['Nelson Agholor'],
 'AgneJa00': ['Jamal Agnew'],
 'AgneRa00': ['Ray Agnew'],
 'AhmeSa01': ['Salvon Ahmed'],
 'AikeKa00': ['Kamar Aiken'],
 'AiyuBr00': ['Brandon Aiyuk'],
 'AjayJa00': ['Jay Ajayi'],
 'AjirSe00': ['Seyi Ajirotutu'],
 'AkerCa00': ['Cam Akers'],
 'AkerLa00': ['Landen Akers'],
 'AkinJo00': ['Jordan Akins'],
 'AkinTo00': ['Tommy Akingbesote'],
 'AlexDa01': ['Darius Alexander'],
 'AlexMa02': ['Maurice Alexander'],
 'AlfoMa00': ['Mario Alford'],
 'AlieMo00': ['Mo AlieCox'],
 'AlixJo00': ['Josh Ali'],
 'AlixRa00': ['Rasheen Ali'],
 'AlleBr00': ['Brandon Allen'],
 'AlleBr05': ['Brael